# Mars Arrival

Goals:
- Course corrections
- B-plane targeting to specific orbit
- Hyperbolic reentry orbit and burn
- Propagation

Idea (thanks Zeb)
- Same orbit but just RAAN change
- Inclination burn

**Assumptions**
- Circular target orbit

In [2]:
%load_ext autoreload
%autoreload 2

In [22]:
import numpy as np
from cesium.constants import (
    DEG_TO_RAD,
    HOUR_TO_SEC,
    KM_TO_M,
    M_TO_KM,
    MIN_TO_SEC,
    RAD_TO_DEG,
)
from cesium.guidance.interplanetary import (
    interplanetary_circ_injection,
)
from cesium.math.base import unit
from cesium.math.time import jd_to_greg
from cesium.orbital.coes import coes_to_rv
from cesium.orbital.interplanetary import hyperbolic_characteristics, sph_of_infl
from cesium.physics.dynamics import two_body
from cesium.plotting import plot_3d
from cesium.world.constants import MU_MARS, MU_SUN, R_MARS
from numpy.lib.npyio import NpzFile
from numpy.linalg import norm
from scipy.integrate import solve_ivp

## Loading trajectory
From `interplanetary.ipynb`

In [ ]:
MISSION_NAME = "birthday-christmas-earlier"
mission: NpzFile = np.load(f"trajectories/{MISSION_NAME}.mission", allow_pickle=False)


def print_mission(mission: NpzFile):
    """Print dates, duration, sampling, units, and stored fields from a mission archive.

    Parameters
    ----------
    mission : NpzFile
        Loaded Mars Courier .mission archive in NumPy NPZ format. Expected
        fields include lambert_departure_jd, lambert_arrival_jd, tof_s,
        time_s, dt_s, frame, position_units, velocity_units, and time_origin.
        Epochs are heliocentric Lambert references, not liftoff or SOI crossings.

    Notes
    -----
    Uses the notebook's MISSION_NAME as the display title. State-array shapes
    and dtypes are listed without changing or closing the loaded archive.
    """
    print(f"Mission: {MISSION_NAME}")
    print("Heliocentric reference times")
    for label, key in (
        ("Lambert departure (Earth reference)", "lambert_departure_jd"),
        ("Lambert arrival (Mars reference)", "lambert_arrival_jd"),
    ):
        print(
            f"{label}: "
            + "{:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f}".format(
                *jd_to_greg(float(mission[key]))
            )
        )
    print(f"Flight duration: {float(mission['tof_s']) / 86400:.4f} days")
    print(
        f"Samples: {mission['time_s'].size} (seconds since {mission['time_origin'].item()})"
    )
    # print(f"Timesteps ranging from {mission['dt_s'].min():.3f}-{mission['dt_s'].max():.3f} s ")
    print(
        f"Frame: {mission['frame'].item()}; positions: {mission['position_units'].item()}; velocities: {mission['velocity_units'].item()}"
    )
    print("\nStored fields:")
    for key in mission.files:
        info = mission[key].dtype

        if mission[key].dtype != float:
            info = '"' + mission[key] + '"'
        elif mission[key].shape == ():
            info = mission[key]

        # info = mission[key].dtype if mission[key].dtype == float else mission[key]
        print(f"  {key:24s} {mission[key].shape!s:14s} {info}")


print_mission(mission)


rv_sat_mci_km = (
    mission["satellite_rv_hci_km"] - mission["mars_rv_hci_km"]
)  # mars centered inertial
r_lambert_end_mci_km = rv_sat_mci_km[-1, 0:3]
v_inf_mci_km = rv_sat_mci_km[-1, 3:6]
lambert_arr_jd = float(mission["lambert_arrival_jd"])

In [74]:
# Desired landing site, basically we want the orbit inclination to be above 18 deg
PERSEVERENCE_LLA = (18.434830485851624, 77.23277091979982, 0)

# HCI right now until I figure out how I want to do this mars-wise
# a,e,i,raan,aop,ta
PARKING_ORBIT_ALT_M = 300e3
PARKING_ORBIT_RAAN_DEG = 20
PARKING_ORBIT_I_DEG = 45
TARGET_COES = (
    R_MARS + PARKING_ORBIT_ALT_M,
    0.0,
    PARKING_ORBIT_I_DEG * DEG_TO_RAD,
    PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
    0,
    0,
)

# Orbit
PARKING_ORBIT_ALT_M = 300e3  # m
mars_soi_km = sph_of_infl(norm(mission["mars_rv_hci_km"][-1, 0:3]), MU_MARS, MU_SUN)

In [115]:
# Parking orbit

rv_target0 = np.concatenate(coes_to_rv(*TARGET_COES, MU_MARS))

t_target = np.linspace(0, 150 * MIN_TO_SEC, 10000)
target_mci = solve_ivp(
    lambda _, s: two_body(s, MU_MARS),
    (t_target[0], t_target[-1]),
    rv_target0,
    t_eval=t_target,
    rtol=1e-9,
    atol=1e-12,
)
assert target_mci.success

rv_target_mci_m = target_mci.y.T

## Arrival Characteristics

In [76]:
# Mars-centered, HCI-aligned axes (not Mars-equatorial axes).
C3_km2, rla, dla = hyperbolic_characteristics(v_inf_mci_km)

PERIAPSE = R_MARS + PARKING_ORBIT_ALT_M
dv_p, ta_asymptote, aiming_radius, e = interplanetary_circ_injection(
    norm(v_inf_mci_km * KM_TO_M), PERIAPSE, MU_MARS, deg=False
)

print("Arrival target (Lambert reference epoch)")
print(f"  v_inf: {norm(v_inf_mci_km):.4f} km/s; C3: {C3_km2:.4f} km²/s²")
print("  Incoming velocity asymptote angles in HCI-aligned axes")
print(
    f"  Longitude (RLA): {rla * RAD_TO_DEG:.4f} deg; latitude (DLA): {dla * RAD_TO_DEG:.4f} deg"
)
print("\nCircular parking orbit / ideal tangential capture burn")
print(f"  Parking altitude: {PARKING_ORBIT_ALT_M * M_TO_KM:.1f} km")
print(f"  Periapsis radius: {PERIAPSE * M_TO_KM:.3f} km from Mars's center")
print(f"  Circular speed: {np.sqrt(MU_MARS / PERIAPSE) * M_TO_KM:.4f} km/s")
print(
    f"  Pre-burn periapsis speed: {np.sqrt(norm(v_inf_mci_km * KM_TO_M) ** 2 + 2 * MU_MARS / PERIAPSE) * M_TO_KM:.4f} km/s"
)
print(f"  Required capture delta-v magnitude: {dv_p * M_TO_KM:.4f} km/s")
print("\nArrival hyperbola")
print(f"  Eccentricity: {e:.6f}")
print(f"  Incoming asymptotic true anomaly: {-ta_asymptote * RAD_TO_DEG:.4f} deg")
print(f"  Aiming radius (impact parameter): {aiming_radius * M_TO_KM:.3f} km")

Arrival target (Lambert reference epoch)
  v_inf: 2.8202 km/s; C3: 7.9534 km²/s²
  Incoming velocity asymptote angles in HCI-aligned axes
  Longitude (RLA): 347.5865 deg; latitude (DLA): -17.7198 deg

Circular parking orbit / ideal tangential capture burn
  Parking altitude: 300.0 km
  Periapsis radius: 3696.000 km from Mars's center
  Circular speed: 3.4041 km/s
  Pre-burn periapsis speed: 5.5793 km/s
  Required capture delta-v magnitude: 2.1752 km/s

Arrival hyperbola
  Eccentricity: 1.686364
  Incoming asymptotic true anomaly: -126.3696 deg
  Aiming radius (impact parameter): 7312.017 km


## Targeting Arrival

In [124]:
class PlanetArrivalTrajectory:
    def __init__(
        self,
        v_inf_hci_km: np.ndarray,
        periapse_m: float,
        target_raan_deg: float,
        target_i_deg: float,
        mu: float,
    ):
        self.v_inf_hci_km = v_inf_hci_km
        self.periapse_m = periapse_m
        # Coplanar circular capture cost and hyperbola geometry.
        self.dv_periapse_m, self.ta_asymptote_rad, self.aiming_radius_m, self.e = (
            interplanetary_circ_injection(
                norm(v_inf_hci_km * KM_TO_M), periapse_m, mu, deg=False
            )
        )
        self.mu = mu
        self.v_vec = unit(v_inf_hci_km)
        target_raan = target_raan_deg * DEG_TO_RAD
        target_i = target_i_deg * DEG_TO_RAD

        # Use the existing element conversion at the ascending node.
        r_target, v_target = coes_to_rv(
            periapse_m, 0.0, target_i, target_raan, 0.0, 0.0, mu
        )
        self.orbit_right_vec = unit(r_target)
        self.orbit_normal = unit(np.cross(r_target, v_target))
        self.orbit_up_vec = np.cross(self.orbit_normal, self.orbit_right_vec)

        # Incoming velocity direction projected into the target plane.
        v_projected = (
            self.v_vec - np.dot(self.v_vec, self.orbit_normal) * self.orbit_normal
        )
        q = norm(v_projected)
        if q < 1 / self.e - 1e-12:
            raise ValueError(
                "No periapsis in this target plane for the chosen v_inf and radius"
            )
        u = unit(v_projected)
        w = np.cross(self.orbit_normal, u)  # along track in target orbit

        # Arrival: p_hat dot v_inf_hat = 1/e. Use basis vectors
        self.theta_rad = np.arccos(np.clip(1 / (self.e * q), -1, 1))
        p_hats = np.array(
            [
                unit(np.cos(self.theta_rad) * u + sign * np.sin(self.theta_rad) * w)
                for sign in (1, -1)
            ]
        )
        self.r_p_candidates_mci_m = periapse_m * p_hats

        # v_inf_hat = p_hat/e + sqrt(1 - 1/e**2) * incoming_periapsis_tangent.
        incoming_tangents = (self.v_vec - p_hats / self.e) / np.sqrt(1 - 1 / self.e**2)
        v_p = np.sqrt(norm(v_inf_hci_km * KM_TO_M) ** 2 + 2 * MU_MARS / periapse_m)
        self.v_p_candidates_mci_m_s = v_p * incoming_tangents
        self.v_capture_candidates_mci_m_s = np.sqrt(MU_MARS / periapse_m) * np.cross(
            self.orbit_normal, p_hats
        )
        self.dv_candidates_mci_m_s = (
            self.v_capture_candidates_mci_m_s - self.v_p_candidates_mci_m_s
        )

        # Keep both branches; select the smaller combined capture/plane-change burn.
        self.selected_index = int(np.argmin(norm(self.dv_candidates_mci_m_s, axis=1)))
        self.r_p_mci_m = self.r_p_candidates_mci_m[self.selected_index]
        self.v_p_mci_m_s = self.v_p_candidates_mci_m_s[self.selected_index]
        self.v_capture_mci_m_s = self.v_capture_candidates_mci_m_s[self.selected_index]
        self.dv_capture_mci_m_s = self.dv_candidates_mci_m_s[self.selected_index]
        self.dv_capture_m_s = norm(self.dv_capture_mci_m_s)

    def launch_that_babyy(self, tspan_s: float = 12 * HOUR_TO_SEC, npoints=100000):
        """Propagates BACKWARDS"""
        rv_mci_0 = np.concatenate((self.r_p_mci_m, -self.v_p_mci_m_s))

        self.t_traj_s = np.linspace(0, tspan_s, npoints)
        departure_mci = solve_ivp(
            lambda _, s: two_body(s, self.mu),
            (self.t_traj_s[0], self.t_traj_s[-1]),
            rv_mci_0,
            t_eval=self.t_traj_s,
            rtol=1e-9,
            atol=1e-12,
        )
        assert departure_mci.success

        self.rv_traj_mci_m = departure_mci.y.T

        return self.rv_traj_mci_m


arrival = PlanetArrivalTrajectory(
    v_inf_mci_km,
    periapse_m=PERIAPSE,
    target_raan_deg=PARKING_ORBIT_RAAN_DEG,
    target_i_deg=PARKING_ORBIT_I_DEG,
    mu=MU_MARS,
)
rv_prop_arr = arrival.launch_that_babyy()
print(
    "Capture delta-v for the two periapsis solutions [km/s]:",
    norm(arrival.dv_candidates_mci_m_s, axis=1) * M_TO_KM,
)
print(
    f"Selected solution: {arrival.selected_index}; capture delta-v: {arrival.dv_capture_m_s * M_TO_KM:.4f} km/s"
)

Capture delta-v for the two periapsis solutions [km/s]: [8.94834429 2.31512624]
Selected solution: 1; capture delta-v: 2.3151 km/s


## Visualization
$
\boxed{
\hat{\mathbf p}
=\frac{1}{eq}\hat{\mathbf u}
\ \pm\
\sqrt{1-\frac{1}{e^2q^2}}\,
(\hat{\mathbf n}\times\hat{\mathbf u})
}$


$
\hat s=\frac{\hat p}{e}+\sqrt{1-\frac1{e^2}}\,\hat t
$

In [127]:
# Translate by Earth's rv at each sample, then rotate HCI into ECI.
sun_direction_hci = -mission["mars_rv_hci_km"][-1, :3]


frames = [
    # {
    #     "name": "HCI",
    #     "location": [0, 0, 0],
    #     "dcm_body_wrt_hci": np.eye(3),
    #     "length": 10000,
    # },
]

vectors = [
    {
        "direction": sun_direction_hci,
        "length": 20000,
        "name": "Sun direction",
        "color": "goldenrod",
        "label": True,
    },
    {
        "direction": arrival.v_vec,
        "length": 10000,
        "name": "v_vec",
        "color": "orange",
        "label": True,
    },
    # {
    #     "direction": arrival.orbit_right_vec,
    #     "length": 10000,
    #     "name": "orbit right vec",
    #     "color": "black",
    #     "label": True,
    # },  # just for visualization
    # {
    #     "direction": arrival.orbit_up_vec,
    #     "length": 10000,
    #     "name": "orbit right vec",
    #     "color": "black",
    #     "label": True,
    # },  # just for visualization
    # {
    #     "direction": arrival.orbit_normal,
    #     "length": 10000,
    #     "name": "orbit right vec",
    #     "color": "black",
    #     "label": True,
    # },  # just for visualization
    # {
    #     "direction": arrival.v_capture_mci_m_s,
    #     "length": 10000,
    #     "name": "V capture direction vec",
    #     "color": "black",
    #     "label": True,
    #     "origin": arrival.r_p_mci_m * M_TO_KM
    # },  # just for visualization
    # {
    #     "direction": arrival.v_p_mci_m_s,
    #     "length": 10000,
    #     "name": "V capture direction vec",
    #     "color": "black",
    #     "label": True,
    #     "origin": arrival.r_p_mci_m * M_TO_KM
    # },  # just for visualization
    # {
    #     "direction": departure.right_vec,
    #     "length": 10000,
    #     "name": "right_vec",
    #     "color": "black",
    #     "label": True,
    # },  # just for visualization
]

points = [
    {
        "name": "Periapse",
        "location": (arrival.r_p_mci_m) * M_TO_KM,
        "color": "red",
    },
    # {
    #     "name": "SOI along v_inf",
    #     "location": -mars_soi_km * unit(v_inf_mci_km),
    #     "color": "green",
    # },
]

view_range = 30000
fig_earth = plot_3d(
    [
        rv_sat_mci_km[:, 0:3],
        rv_target_mci_m[:, 0:3] * M_TO_KM,
        rv_prop_arr[:, 0:3] * M_TO_KM,
    ],
    [mission["time_s"], t_target, arrival.t_traj_s],
    labels=["Lambert Arrival Direction", "Target Orbit", "Propagated from V_inf"],
    trace_colors=["purple", "black", "green"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    frames=frames,
    vectors=vectors,
    points=points,
    title=f"Mars arrival — {view_range} km view",
    units="km",
    azim=80,
    elev=30,
    show_end=False,
    show=False,
)
fig_earth.update_scenes(
    xaxis_range=[-view_range, view_range],  # km
    yaxis_range=[-view_range, view_range],  # km
    zaxis_range=[-view_range, view_range],  # km
)
fig_earth.show()